# Week 10: Performance Intuition — PHASE 6: Scaling Your Pipeline
*Core Mastery: "I can measure and compare the performance of different approaches"*

*Computer Programming II | 5 Hours | Dr. Arif Solmaz*

## 🎯 Learning Objectives

By the end of this week, you will be able to:

1. Measure execution time using `time.time()` and `time.perf_counter()`
2. Build a reusable `benchmark()` helper function
3. Compare the speed of loops, list comprehensions, and NumPy operations
4. Design scaling experiments that vary input size systematically
5. Recognize O(n) vs O(n²) behaviour from empirical measurements
6. Profile a multi-stage pipeline to identify bottlenecks
7. Create formatted benchmark reports with tables
8. Plot performance curves to visualize scaling behaviour
9. Make informed decisions about when optimization is worth the effort
10. Apply performance measurement to engineering data pipelines

## 🎯 Core Mastery Connection

PHASE 6: Scaling Your Pipeline — This week's topic directly supports the course's core mastery goal: *"I can measure and compare the performance of different approaches"*.

When you process large sensor datasets or engineering simulations, the difference between a 1-second and a 10-minute run matters. This week you will learn to **measure first, then decide** — the cardinal rule of performance work.

| Skill | Why It Matters |
|---|---|
| Timing code | Know *exactly* how long each step takes |
| Benchmarking | Compare alternatives fairly |
| Scaling experiments | Predict how code behaves on larger data |
| Bottleneck detection | Focus effort where it matters most |
| Optimization decisions | Avoid wasting time on code that is already fast |

---
## Part 1: Why Performance Matters

### The Cost of Slow Code

In engineering applications, data volumes grow fast. A temperature logger running at 10 Hz for a week produces over **6 million readings**. A structural simulation grid with 1000 × 1000 cells needs to update every time-step. If your code is slow, you wait — and waiting is expensive.

| Scenario | Data Size | Slow Code (loop) | Fast Code (NumPy) |
|---|---|---|---|
| Sensor batch | 10 000 rows | 0.8 s | 0.003 s |
| Simulation grid | 1 000 000 cells | 85 s | 0.09 s |
| Image batch | 500 images × 1 MP | 12 min | 4 s |

### The Golden Rule

> **Measure first, optimize second.** Never guess where the bottleneck is — time it.

Most beginners optimize the wrong part of their code. Measurement removes guesswork.

### What We Will Cover

1. How to time a block of code
2. How to build a reusable benchmark helper
3. How to compare two approaches fairly
4. How to run scaling experiments
5. How to profile a multi-stage pipeline
6. How to decide whether to optimize

**Figure 10.1** — A simple timing example

In [ ]:
import time

# Task: sum the first 1 million integers using a loop
n = 1_000_000

start = time.time()
total = 0
for i in range(n):
    total += i
elapsed = time.time() - start

print(f"Sum = {total:,}")
print(f"Loop took {elapsed:.4f} seconds")

**Figure 10.2** — The same task with a built-in function

In [ ]:
import time

n = 1_000_000

start = time.time()
total = sum(range(n))
elapsed = time.time() - start

print(f"Sum = {total:,}")
print(f"sum(range(...)) took {elapsed:.6f} seconds")
print("Built-in functions are implemented in C — much faster than Python loops!")

---
## Part 2: Measuring Time — `time.time()` Basics

### Two Timing Functions

| Function | Resolution | Best For |
|---|---|---|
| `time.time()` | ~1 ms | General wall-clock timing |
| `time.perf_counter()` | ~1 µs | Precise micro-benchmarks |

### The Pattern

```python
import time
start = time.perf_counter()
# ... code to measure ...
elapsed = time.perf_counter() - start
print(f"Elapsed: {elapsed:.6f} s")
```

### Tips

- Always time the **same input** when comparing two approaches
- Run the measurement **multiple times** and report the average
- Avoid running other heavy tasks during measurement
- For very fast operations (< 1 ms), use `perf_counter` and repeat many times

**Figure 10.3** — Comparing `time.time()` vs `time.perf_counter()`

In [ ]:
import time

data = list(range(500_000))

# time.time()
t0 = time.time()
_ = [x**2 for x in data]
t1 = time.time()
print(f"time.time()         : {t1 - t0:.6f} s")

# time.perf_counter()
t0 = time.perf_counter()
_ = [x**2 for x in data]
t1 = time.perf_counter()
print(f"time.perf_counter() : {t1 - t0:.6f} s")

print("\nperf_counter has higher resolution — use it for short operations.")

**Figure 10.4** — Averaging multiple runs for stable results

In [ ]:
import time

def average_time(func, *args, runs=5):
    """Return average execution time over several runs."""
    times = []
    for _ in range(runs):
        t0 = time.perf_counter()
        func(*args)
        t1 = time.perf_counter()
        times.append(t1 - t0)
    return sum(times) / len(times)

data = list(range(500_000))

avg = average_time(sorted, data, runs=5)
print(f"sorted() average over 5 runs: {avg:.6f} s")

---
## Part 3: The `benchmark()` Helper Function

A reusable benchmark function saves you from writing the same timing boilerplate every time. Let's build one step by step.

### Design Goals

| Feature | Why |
|---|---|
| Accept any callable | Benchmark *any* function |
| Multiple runs | Get stable averages |
| Return dict with stats | Easy to compare later |
| Pretty-print option | Quick visual feedback |

### Building Blocks

1. Accept `func` and `*args`
2. Time each run
3. Compute min, max, mean
4. Optionally print a summary

**Figure 10.5** — Building the `benchmark()` helper

In [ ]:
import time

def benchmark(func, *args, runs=7, label=None, verbose=True):
    """Time func(*args) over multiple runs and return stats dict."""
    times = []
    for _ in range(runs):
        t0 = time.perf_counter()
        result = func(*args)
        t1 = time.perf_counter()
        times.append(t1 - t0)

    stats = {
        "label": label or func.__name__,
        "runs": runs,
        "min": min(times),
        "max": max(times),
        "mean": sum(times) / len(times),
        "times": times,
    }

    if verbose:
        print(f"⏱  {stats['label']:30s}  "
              f"mean={stats['mean']:.6f}s  "
              f"min={stats['min']:.6f}s  "
              f"max={stats['max']:.6f}s  "
              f"({runs} runs)")

    return stats

# Demo
data = list(range(200_000))
benchmark(sorted, data, label="sorted(200k)")

**Figure 10.6** — Using benchmark to compare `sum()` vs manual loop

In [ ]:
def loop_sum(data):
    total = 0
    for x in data:
        total += x
    return total

data = list(range(500_000))

s1 = benchmark(loop_sum, data, label="for-loop sum")
s2 = benchmark(sum, data, label="built-in sum()")

speedup = s1["mean"] / s2["mean"]
print(f"\n🚀 Built-in sum() is {speedup:.1f}x faster than a for-loop")

**Figure 10.7** — Extending benchmark to compare a list of functions

In [ ]:
def compare(funcs_and_args, runs=7):
    """Benchmark several (label, func, args) tuples and print a table."""
    results = []
    for label, func, args in funcs_and_args:
        stats = benchmark(func, *args, runs=runs, label=label, verbose=False)
        results.append(stats)

    # Print comparison table
    fastest = min(results, key=lambda r: r["mean"])["mean"]
    print(f"{'Label':35s} {'Mean (s)':>12s} {'Relative':>10s}")
    print("-" * 60)
    for r in sorted(results, key=lambda r: r["mean"]):
        rel = r["mean"] / fastest
        bar = "█" * int(rel * 10)
        print(f"{r['label']:35s} {r['mean']:12.6f} {rel:9.1f}x  {bar}")
    return results

data = list(range(300_000))

compare([
    ("for-loop sum",   loop_sum, (data,)),
    ("built-in sum()", sum,      (data,)),
    ("math.fsum()",    __import__('math').fsum, (data,)),
])

---
## Part 4: Comparing Two Solutions

### Fair Comparison Rules

When comparing two approaches, keep everything else the same:

| Variable | Must Be Identical? |
|---|---|
| Input data | ✅ Yes — same list / array |
| Input size | ✅ Yes — same n |
| Number of runs | ✅ Yes |
| Machine load | ⚠️ Minimize background tasks |
| Python version | ✅ Yes (same Colab runtime) |

### Common Comparison Pairs

| Approach A | Approach B | Typical Speedup |
|---|---|---|
| for-loop | List comprehension | 1.2–2x |
| List comprehension | `map()` + function | ~1x |
| Python loop | NumPy vectorized | 10–100x |
| Repeated string `+=` | `"".join()` | 5–50x |
| `dict` lookup | `list.index()` | 10–1000x |

**Figure 10.8** — Loop vs list comprehension vs `map()`

In [ ]:
import time

data = list(range(500_000))

def via_loop(data):
    result = []
    for x in data:
        result.append(x ** 2)
    return result

def via_comp(data):
    return [x ** 2 for x in data]

def via_map(data):
    return list(map(lambda x: x ** 2, data))

compare([
    ("for-loop + append", via_loop, (data,)),
    ("list comprehension", via_comp, (data,)),
    ("map() + lambda",     via_map,  (data,)),
])

**Figure 10.9** — String concatenation: `+=` vs `join()`

In [ ]:
def concat_plus(words):
    result = ""
    for w in words:
        result += w + " "
    return result.strip()

def concat_join(words):
    return " ".join(words)

words = [f"kelime_{i}" for i in range(50_000)]

compare([
    ("str += loop",  concat_plus, (words,)),
    ("' '.join()",   concat_join, (words,)),
])

**Figure 10.10** — Dict lookup vs list search

In [ ]:
import random

n = 50_000
keys = [f"sensor_{i}" for i in range(n)]
values = [random.uniform(20, 30) for _ in range(n)]
lookup_dict = dict(zip(keys, values))
lookup_list = list(zip(keys, values))

targets = random.choices(keys, k=1000)

def search_dict(targets, d):
    return [d[k] for k in targets]

def search_list(targets, lst):
    result = []
    for k in targets:
        for key, val in lst:
            if key == k:
                result.append(val)
                break
    return result

compare([
    ("dict lookup (1000 queries)",  search_dict, (targets, lookup_dict)),
    ("list scan  (1000 queries)",   search_list, (targets, lookup_list[:2000])),
], runs=3)

---
## Part 5: Understanding O(n) vs O(n²) with Experiments

### Big-O Intuition

| Notation | Name | Doubles Input → Time ... |
|---|---|---|
| O(1) | Constant | Stays the same |
| O(n) | Linear | Doubles |
| O(n log n) | Log-linear | Slightly more than doubles |
| O(n²) | Quadratic | Quadruples |
| O(2ⁿ) | Exponential | Explodes |

### How to Detect Big-O Empirically

1. Time your function for n = 1000, 2000, 4000, 8000, …
2. Compute the **ratio** of consecutive times
3. If ratio ≈ 2 → O(n). If ratio ≈ 4 → O(n²). If ratio ≈ 8 → O(n³).

### Why This Matters

An O(n²) algorithm that runs in 0.01 s on 1000 items will take **100 s** on 100 000 items — that's a 10 000x slowdown for a 100x increase in data!

**Figure 10.11** — Empirical scaling experiment: O(n) vs O(n²)

In [ ]:
import time

def linear_search(data, target):
    """O(n) — scan the list once."""
    for x in data:
        if x == target:
            return True
    return False

def has_duplicate_naive(data):
    """O(n²) — compare every pair."""
    for i in range(len(data)):
        for j in range(i + 1, len(data)):
            if data[i] == data[j]:
                return True
    return False

sizes = [1000, 2000, 4000, 8000]
print(f"{'n':>8s}  {'O(n) (s)':>12s}  {'O(n²) (s)':>12s}  {'Ratio O(n)':>12s}  {'Ratio O(n²)':>12s}")
print("-" * 65)

prev_lin = prev_quad = None
for n in sizes:
    data = list(range(n))
    target = -1  # worst case: not found

    t0 = time.perf_counter()
    for _ in range(100):
        linear_search(data, target)
    t_lin = (time.perf_counter() - t0) / 100

    t0 = time.perf_counter()
    has_duplicate_naive(data)
    t_quad = time.perf_counter() - t0

    r_lin  = f"{t_lin / prev_lin:.2f}x" if prev_lin else "—"
    r_quad = f"{t_quad / prev_quad:.2f}x" if prev_quad else "—"
    print(f"{n:8d}  {t_lin:12.6f}  {t_quad:12.6f}  {r_lin:>12s}  {r_quad:>12s}")
    prev_lin, prev_quad = t_lin, t_quad

print("\n📊 O(n) ratio ≈ 2x when n doubles.  O(n²) ratio ≈ 4x when n doubles.")

**Figure 10.12** — Visualizing scaling with matplotlib

In [ ]:
import time
import matplotlib.pyplot as plt

sizes = [500, 1000, 2000, 4000, 8000, 16000]
times_linear = []
times_quad   = []

for n in sizes:
    data = list(range(n))

    # O(n)
    t0 = time.perf_counter()
    for _ in range(50):
        linear_search(data, -1)
    times_linear.append((time.perf_counter() - t0) / 50)

    # O(n²)
    t0 = time.perf_counter()
    has_duplicate_naive(data)
    times_quad.append(time.perf_counter() - t0)

fig, axes = plt.subplots(1, 2, figsize=(12, 4))

axes[0].plot(sizes, times_linear, 'o-', color='steelblue')
axes[0].set_title("O(n) — Linear Search")
axes[0].set_xlabel("Input size n")
axes[0].set_ylabel("Time (s)")

axes[1].plot(sizes, times_quad, 's-', color='crimson')
axes[1].set_title("O(n²) — Naive Duplicate Check")
axes[1].set_xlabel("Input size n")
axes[1].set_ylabel("Time (s)")

plt.tight_layout()
plt.show()

**Figure 10.13** — Faster duplicate check with a set: O(n)

In [ ]:
def has_duplicate_set(data):
    """O(n) — use a set for fast membership test."""
    seen = set()
    for x in data:
        if x in seen:
            return True
        seen.add(x)
    return False

# Compare at n = 8000
data = list(range(8000))

compare([
    ("naive O(n²)", has_duplicate_naive, (data,)),
    ("set O(n)",    has_duplicate_set,   (data,)),
], runs=3)

---
## Part 6: Profiling a Multi-Stage Pipeline

### Pipeline Profiling Strategy

Real-world data pipelines have multiple stages. To optimize, you need to find the **bottleneck** — the slowest stage.

| Step | Action |
|---|---|
| 1 | Identify all stages |
| 2 | Time each stage independently |
| 3 | Compute percentage of total time |
| 4 | Focus optimization on the biggest percentage |

### Ahmet's Sensor Pipeline

Ahmet is processing temperature data from 50 sensors over a month (1.5 million readings):

1. **Load** — read CSV data
2. **Clean** — remove invalid readings
3. **Compute** — calculate daily averages
4. **Format** — build output report

**Figure 10.14** — Profiling a 4-stage pipeline

In [ ]:
import time
import random

# Simulate 500,000 sensor readings
random.seed(42)
raw_data = [
    {"sensor": f"S{random.randint(1,50):02d}",
     "temp": random.gauss(22, 5) if random.random() > 0.05 else -999,
     "day": random.randint(1, 30)}
    for _ in range(500_000)
]

def stage_load(data):
    """Simulate loading — just copy the data."""
    return [dict(row) for row in data]

def stage_clean(data):
    """Remove invalid readings (-999)."""
    return [row for row in data if row["temp"] != -999]

def stage_compute(data):
    """Compute daily average per sensor."""
    from collections import defaultdict
    buckets = defaultdict(list)
    for row in data:
        key = (row["sensor"], row["day"])
        buckets[key].append(row["temp"])
    return {k: sum(v)/len(v) for k, v in buckets.items()}

def stage_format(averages):
    """Build a summary report string."""
    lines = []
    for (sensor, day), avg in sorted(averages.items()):
        lines.append(f"{sensor} Day {day:2d}: {avg:6.2f}°C")
    return "\n".join(lines)

# Profile each stage
stages = [
    ("1-Load",    stage_load,    (raw_data,)),
    ("2-Clean",   stage_clean,   None),
    ("3-Compute", stage_compute, None),
    ("4-Format",  stage_format,  None),
]

profile = {}
result = raw_data
for name, func, args in stages:
    if args is None:
        args = (result,)
    t0 = time.perf_counter()
    result = func(*args)
    elapsed = time.perf_counter() - t0
    profile[name] = elapsed

total = sum(profile.values())
print(f"{'Stage':15s} {'Time (s)':>10s} {'%':>8s}  Bar")
print("-" * 55)
for name, t in profile.items():
    pct = t / total * 100
    bar = "█" * int(pct / 2)
    print(f"{name:15s} {t:10.4f} {pct:7.1f}%  {bar}")
print(f"{'TOTAL':15s} {total:10.4f}")
print(f"\n🔍 Bottleneck: {max(profile, key=profile.get)}")

**Figure 10.15** — Pie chart of pipeline stages

In [ ]:
import matplotlib.pyplot as plt

labels = list(profile.keys())
times  = list(profile.values())
colors = ['#4e79a7', '#f28e2b', '#e15759', '#76b7b2']

fig, ax = plt.subplots(figsize=(6, 6))
ax.pie(times, labels=labels, autopct='%1.1f%%', colors=colors,
       startangle=90, textprops={'fontsize': 12})
ax.set_title("Pipeline Stage Time Distribution", fontsize=14)
plt.show()

---
## Part 7: Loop vs NumPy — Systematic Comparison

### Why NumPy is Fast

NumPy operations are:
- Written in C and Fortran under the hood
- Vectorized: operate on entire arrays at once
- Cache-friendly: data stored in contiguous memory

| Operation | Python Loop | NumPy |
|---|---|---|
| Element-wise multiply | `[a*b for a,b in zip(x,y)]` | `x * y` |
| Sum | `sum(data)` | `np.sum(data)` |
| Mean | `sum(d)/len(d)` | `np.mean(d)` |
| Filter | `[x for x in d if x>0]` | `d[d > 0]` |
| Square root | `[math.sqrt(x) for x in d]` | `np.sqrt(d)` |

### When to Use NumPy

- Processing **numeric arrays** with > 10 000 elements
- Performing the **same operation** on every element
- Working with **multi-dimensional** data (matrices, images)

### When Pure Python is Fine

- Small data (< 1000 items)
- Complex logic with many branches per element
- String processing, dict manipulation

**Figure 10.16** — NumPy vs loop: element-wise operations

In [ ]:
import numpy as np
import time

n = 1_000_000
py_list = list(range(n))
np_array = np.arange(n, dtype=np.float64)

# Python loop: square each element
def loop_square(data):
    return [x**2 for x in data]

# NumPy: square each element
def numpy_square(arr):
    return arr ** 2

compare([
    ("Python list comp (1M)", loop_square,  (py_list,)),
    ("NumPy vectorized (1M)", numpy_square, (np_array,)),
])

**Figure 10.17** — Scaling comparison: Python vs NumPy across sizes

In [ ]:
import time
import numpy as np

sizes = [1_000, 10_000, 100_000, 500_000, 1_000_000]
py_times = []
np_times = []

for n in sizes:
    py_data = list(range(n))
    np_data = np.arange(n, dtype=np.float64)

    # Python
    t0 = time.perf_counter()
    _ = [x**2 for x in py_data]
    py_times.append(time.perf_counter() - t0)

    # NumPy
    t0 = time.perf_counter()
    _ = np_data ** 2
    np_times.append(time.perf_counter() - t0)

print(f"{'n':>12s}  {'Python (s)':>12s}  {'NumPy (s)':>12s}  {'Speedup':>10s}")
print("-" * 52)
for i, n in enumerate(sizes):
    sp = py_times[i] / np_times[i] if np_times[i] > 0 else float('inf')
    print(f"{n:12,d}  {py_times[i]:12.6f}  {np_times[i]:12.6f}  {sp:9.1f}x")

**Figure 10.18** — Plotting Python vs NumPy scaling

In [ ]:
import matplotlib.pyplot as plt

fig, ax = plt.subplots(figsize=(8, 5))
ax.plot(sizes, py_times, 'o-', label='Python loop', color='crimson')
ax.plot(sizes, np_times, 's-', label='NumPy vectorized', color='steelblue')
ax.set_xlabel('Input size n')
ax.set_ylabel('Time (seconds)')
ax.set_title('Element-wise Square: Python Loop vs NumPy')
ax.legend()
ax.set_xscale('log')
ax.set_yscale('log')
ax.grid(True, alpha=0.3)
plt.tight_layout()
plt.show()

**Figure 10.19** — NumPy filtering vs list comprehension

In [ ]:
import numpy as np

n = 1_000_000
np_data = np.random.uniform(-50, 50, n)
py_data = np_data.tolist()

def py_filter(data):
    return [x for x in data if x > 0]

def np_filter(arr):
    return arr[arr > 0]

compare([
    ("Python list comp filter", py_filter, (py_data,)),
    ("NumPy boolean indexing",  np_filter, (np_data,)),
])

---
## Part 8: When to Optimize and When Not To

### The Three Rules of Optimization

1. **Don't optimize yet.** Get correct code first.
2. **Measure before changing anything.** Find the real bottleneck.
3. **Optimize the bottleneck, then re-measure.** Verify improvement.

### Decision Flowchart

```
Is the code correct?
  └─ No  → Fix bugs first, don't optimize broken code
  └─ Yes → Is it fast enough for your use case?
              └─ Yes → STOP. Don't optimize.
              └─ No  → Measure to find the bottleneck
                         └─ Optimize the bottleneck
                         └─ Re-measure to verify improvement
```

### Common Optimization Strategies (Ranked by Effort)

| Strategy | Effort | Typical Speedup |
|---|---|---|
| Use built-in functions | Low | 2–5x |
| Use list comprehensions | Low | 1.2–2x |
| Use NumPy for numeric work | Medium | 10–100x |
| Cache expensive computations | Medium | 2–10x |
| Use better algorithms (O(n²) → O(n)) | High | 10–10000x |
| Use multiprocessing | High | 2–8x (CPU-bound) |

### Premature Optimization

> "Premature optimization is the root of all evil." — Donald Knuth

This means: **do not** spend hours making code 5% faster if it only runs once a week. Focus on **readability** and **correctness** first.

**Figure 10.20** — Decision helper: should you optimize?

In [ ]:
def should_optimize(current_time_s, target_time_s, runs_per_day, dev_hours_estimate):
    """Help decide whether optimization is worth the effort."""
    time_saved_per_run = current_time_s - target_time_s
    if time_saved_per_run <= 0:
        print("✅ Already fast enough! No optimization needed.")
        return False

    daily_savings_s = time_saved_per_run * runs_per_day
    days_to_break_even = (dev_hours_estimate * 3600) / daily_savings_s

    print(f"Current time     : {current_time_s:.2f}s per run")
    print(f"Target time      : {target_time_s:.2f}s per run")
    print(f"Savings per run  : {time_saved_per_run:.2f}s")
    print(f"Runs per day     : {runs_per_day}")
    print(f"Daily savings    : {daily_savings_s:.0f}s ({daily_savings_s/60:.1f} min)")
    print(f"Dev effort       : {dev_hours_estimate}h")
    print(f"Break-even       : {days_to_break_even:.0f} days")
    print()

    if days_to_break_even < 30:
        print("✅ Worth optimizing — break-even within a month.")
        return True
    elif days_to_break_even < 180:
        print("⚠️  Borderline — consider if you have time.")
        return True
    else:
        print("❌ Probably not worth it — break-even takes too long.")
        return False

# Example: Elif's batch processing pipeline
print("=== Elif's Sensor Pipeline ===")
should_optimize(
    current_time_s=45,
    target_time_s=5,
    runs_per_day=10,
    dev_hours_estimate=4
)

print()

# Example: One-time data conversion
print("=== One-Time Data Migration ===")
should_optimize(
    current_time_s=300,
    target_time_s=60,
    runs_per_day=0.01,   # once every 100 days
    dev_hours_estimate=8
)

**Figure 10.21** — Putting it all together: a performance report

In [ ]:
import time
import numpy as np

def generate_performance_report(data_py, data_np):
    """Generate a formatted performance report comparing approaches."""
    tests = {
        "Sum":     (sum, (data_py,), np.sum, (data_np,)),
        "Mean":    (lambda d: sum(d)/len(d), (data_py,), np.mean, (data_np,)),
        "Std Dev": (lambda d: (sum((x - sum(d)/len(d))**2 for x in d)/len(d))**0.5, (data_py,),
                    np.std, (data_np,)),
        "Filter>0":(lambda d: [x for x in d if x > 0], (data_py,),
                    lambda a: a[a > 0], (data_np,)),
    }

    print("=" * 70)
    print(f"  PERFORMANCE REPORT — n = {len(data_py):,}")
    print("=" * 70)
    print(f"{'Operation':15s} {'Python (ms)':>12s} {'NumPy (ms)':>12s} {'Speedup':>10s}")
    print("-" * 55)

    for name, (py_func, py_args, np_func, np_args) in tests.items():
        t0 = time.perf_counter()
        py_func(*py_args)
        py_t = (time.perf_counter() - t0) * 1000

        t0 = time.perf_counter()
        np_func(*np_args)
        np_t = (time.perf_counter() - t0) * 1000

        sp = py_t / np_t if np_t > 0 else float('inf')
        print(f"{name:15s} {py_t:11.3f}  {np_t:11.3f}  {sp:9.1f}x")

    print("=" * 70)

n = 200_000
py_data = [float(x) for x in range(-n//2, n//2)]
np_data = np.array(py_data)
generate_performance_report(py_data, np_data)

---
## Exercises

Complete the exercises below. Write your code in the provided code cells.

> **Each exercise is a problem. Think about the STEPS before you code.** What data do you need? What calculations? What output? Plan your steps first, then translate them into Python.

> Run each cell after writing your solution to check if it works!

### Exercise 1: Time Two Approaches (Easy)

Burak has two functions to compute the sum of squares of numbers from 1 to n. Time both and print which is faster and by how much.

```python
def sum_sq_loop(n):
    total = 0
    for i in range(1, n+1):
        total += i**2
    return total

def sum_sq_comp(n):
    return sum(i**2 for i in range(1, n+1))
```

Use `n = 500_000`. Print the time for each approach and the speedup ratio.

<details>
<summary>💡 Hint</summary>
Use `time.perf_counter()` before and after each function call.
</details>

In [ ]:
# ✏️ [EX1]


### Exercise 2: Build a `benchmark()` Helper (Easy–Medium)

Create your own `benchmark(func, *args, runs=5)` function that:
1. Runs `func(*args)` the given number of times
2. Returns a dictionary with keys: `"label"`, `"mean"`, `"min"`, `"max"`, `"all_times"`
3. Prints a one-line summary

Test it by benchmarking `sorted()` on a list of 100,000 random integers.

<details>
<summary>💡 Hint</summary>
Use a list to collect times from each run, then compute statistics.
</details>

In [ ]:
# ✏️ [EX2]


### Exercise 3: Find the Bottleneck (Medium)

Zeynep has a 3-stage pipeline for processing student exam scores:
1. **Generate** — create 200,000 random scores between 0 and 100
2. **Filter** — keep only scores above 50
3. **Statistics** — compute mean, median, and standard deviation

Time each stage, compute the percentage of total time, and identify the bottleneck. Print a formatted table.

<details>
<summary>💡 Hint</summary>
For median, sort the filtered list and pick the middle element.
</details>

In [ ]:
# ✏️ [EX3]


### Exercise 4: Scaling Experiment (Medium)

Run a function for input sizes n = 1000, 2000, 4000, 8000, 16000 and record the time for each. Compute the ratio between consecutive sizes. Determine whether the function is O(n) or O(n²).

Use this function:
```python
def mystery(data):
    return sorted(data)
```

Print a table with columns: n, time, ratio. What is the Big-O?

<details>
<summary>💡 Hint</summary>
`sorted()` is O(n log n). The ratio should be slightly more than 2x when n doubles.
</details>

In [ ]:
# ✏️ [EX4]


### Exercise 5: O(n) vs O(n²) Demonstration (Medium)

Write two functions that both check whether a list contains any duplicates:
- `has_dup_naive(data)` — O(n²) nested loop approach
- `has_dup_set(data)` — O(n) set-based approach

Time both for sizes n = 500, 1000, 2000, 4000. Print a comparison table showing that the naive approach slows down ~4x when n doubles while the set approach slows down ~2x.

<details>
<summary>💡 Hint</summary>
Use `list(range(n))` as input (no duplicates = worst case for both).
</details>

In [ ]:
# ✏️ [EX5]


### Exercise 6: Pipeline Profiling (Medium)

Mehmet has a data pipeline for processing earthquake magnitude data:
1. **Load** — generate 300,000 random magnitudes (0.0–9.0)
2. **Clean** — remove values outside [0.5, 8.5]
3. **Bin** — count magnitudes in bins: [0–2), [2–4), [4–6), [6–9)
4. **Report** — create a formatted string report with counts and percentages

Profile all 4 stages. Print a table with stage name, time, and percentage. Draw a simple bar chart using `"█"` characters.

<details>
<summary>💡 Hint</summary>
Use `random.uniform(0, 9)` to generate data. For binning, use if-elif chains or integer division.
</details>

In [ ]:
# ✏️ [EX6]


### Exercise 7: NumPy Speedup Measurement (Medium)

Compare Python loops vs NumPy for these three operations on an array of 1,000,000 floats:
1. Compute the square root of each element
2. Find the mean
3. Count elements greater than 0.5

Print a formatted table with operation name, Python time, NumPy time, and speedup.

<details>
<summary>💡 Hint</summary>
Use `math.sqrt` in a loop vs `np.sqrt`. Use `sum()/len()` vs `np.mean()`. Use a list comp with `if` vs boolean indexing.
</details>

In [ ]:
# ✏️ [EX7]


### Exercise 8: Formatted Performance Report (Medium)

Create a function `perf_report(operations, n)` that:
- Takes a list of `(name, py_func, np_func)` tuples and a data size `n`
- Generates random data of size `n`
- Times each Python and NumPy version
- Prints a nicely formatted report with borders, alignment, and speedup ratios
- Includes a "Winner" column showing 🐍 for Python or 🚀 for NumPy

Test with at least 4 operations.

<details>
<summary>💡 Hint</summary>
Use f-string formatting with width specifiers like `{name:20s}` for alignment.
</details>

In [ ]:
# ✏️ [EX8]


### Exercise 9: Optimization Decision (Easy–Medium)

Selin's image processing pipeline takes 120 seconds per batch. She runs it 5 times per day. She estimates 6 hours of development to bring it down to 20 seconds.

Write a function that computes:
- Time saved per day (in minutes)
- Days to break even
- Whether optimization is worthwhile (< 30 days = yes)

Print a clear recommendation.

<details>
<summary>💡 Hint</summary>
Break-even days = dev_hours × 3600 / daily_savings_seconds.
</details>

In [ ]:
# ✏️ [EX9]


### Exercise 10: Benchmark with Visualization (Medium–Challenge)

Extend your benchmark function to accept a list of `(label, func, args)` tuples and produce:
1. A printed comparison table
2. A horizontal bar chart using matplotlib showing mean times

Test with 3 different sorting approaches on 200,000 integers: `sorted()`, `list.sort()` (copy first), and a manual bubble sort on 5,000 items.

<details>
<summary>💡 Hint</summary>
Use `plt.barh()` for horizontal bars. For bubble sort, use a much smaller n to avoid waiting too long.
</details>

In [ ]:
# ✏️ [EX10]


### Exercise 11: Memory vs Speed Tradeoff (Medium)

Compare two approaches to counting word frequencies in a large text:
- **Approach A**: Use a dictionary manually (`for word in words: d[word] = d.get(word, 0) + 1`)
- **Approach B**: Use `collections.Counter`

Generate 500,000 random words from a vocabulary of 1000 words. Time both approaches. Also measure memory using `sys.getsizeof()` on the results.

<details>
<summary>💡 Hint</summary>
Use `random.choices(vocabulary, k=500_000)` to generate words.
</details>

In [ ]:
# ✏️ [EX11]


### Exercise 12: Scaling Plot (Challenge)

Create a function that:
1. Takes a function and a list of input sizes
2. Times the function at each size
3. Plots time vs size on a log-log scale
4. Fits a power law (time ∝ n^k) and prints the estimated exponent k

Test with:
- `sorted()` — expected k ≈ 1 (actually n log n)
- A nested loop function — expected k ≈ 2

<details>
<summary>💡 Hint</summary>
On a log-log scale, slope = exponent. Use `np.polyfit(np.log(sizes), np.log(times), 1)` to estimate k.
</details>

In [ ]:
# ✏️ [EX12]


### Exercise 13: Real Pipeline Optimization (Challenge)

Deniz has this pipeline for processing GPS coordinate data:
1. Generate 100,000 (lat, lon) pairs as a list of tuples
2. Compute distances from a reference point using the Euclidean formula
3. Filter points within 50 km
4. Sort by distance
5. Format top 10 results

Write TWO versions: one using pure Python loops, one using NumPy arrays. Profile both pipelines stage by stage and create a side-by-side comparison report.

<details>
<summary>💡 Hint</summary>
For NumPy, store lat and lon as separate arrays. Use `np.sqrt((lat-ref_lat)**2 + (lon-ref_lon)**2)` for vectorized distance.
</details>

In [ ]:
# ✏️ [EX13]


### Exercise 14: Caching Speedup (Medium)

Demonstrate how caching (memoization) speeds up a recursive Fibonacci function. Time `fib(30)` without caching and with `functools.lru_cache`. Show the speedup.

Also time an iterative version and compare all three.

<details>
<summary>💡 Hint</summary>
The naive recursive fib is O(2^n). With caching it becomes O(n). The iterative version is also O(n).
</details>

In [ ]:
# ✏️ [EX14]


### Exercise 15: Complete Performance Analysis Report (Challenge)

Write a comprehensive performance analysis script that:
1. Defines a data pipeline with at least 4 stages
2. Profiles each stage for 3 different input sizes (10K, 100K, 500K)
3. Creates a table showing how each stage scales
4. Identifies the bottleneck at each size
5. Produces a matplotlib figure with 2 subplots:
   - Left: stacked bar chart of stage times per size
   - Right: line plot showing how the bottleneck stage scales
6. Prints a written recommendation about which stage to optimize

Use Elif's environmental sensor data scenario (temperature, humidity, pressure readings).

<details>
<summary>💡 Hint</summary>
Use `plt.bar()` with `bottom` parameter for stacked bars. Track the bottleneck name at each size.
</details>

In [ ]:
# ✏️ [EX15]


---
### 🌉 Bridge to Next Week

Next week we will learn about **project structure and imports — organizing code across files and importing helper functions in Colab** — building on what you learned this week.

Keep practicing and see you in Week 11!